# MHD on patches

`MHD2D` solves ideal MHD in the plane, with all three components of the velocity and the
magnetic field, on the patches of a patchwork hierarchy. The model has no grid of its own:
a uniform grid is a hierarchy of one level, and refined patches are patches of finer
levels. The in-plane field is the curl of a vector potential $A_z$ on the nodes, so its
divergence is zero to round-off in every cell, across the edges of the patches too. This
tutorial runs the Orszag–Tang vortex, checks the order on an Alfvén wave and moves a
magnetic loop across a refined patch. Every case runs in seconds.

In [ ]:
import numpy as np
import plasma_plots
from patchwork import AMRHierarchy

from carafe import MHD2D

plasma_plots.set_backend("plotly")
plasma_plots.set_math("latex");

## 1. The Orszag–Tang vortex

A periodic square with a smooth vortex in the velocity and the field. Shocks form and
interact by $t = 0.5$; the pressure shows them. `MHD2D.orszag_tang` sets the standard
initial state (Gardiner and Stone 2005).

In [ ]:
vortex = MHD2D.orszag_tang(64)
steps = vortex.run(0.5)
data = vortex.dataset()
data.pressure.plasma.plot.slice(x="x", y="y")

The divergence of the field (times the cell size) and the change of the conserved totals
stay at round-off:

In [ ]:
print(f"{steps} steps, max |div B| dx = {vortex.divergence():.1e}")
print(
    f"pressure from {float(data.pressure.min()):.3f} to {float(data.pressure.max()):.3f}"
)

## 2. Order of accuracy

A circularly polarized Alfvén wave (Tóth 2000) is an exact nonlinear solution: it travels
obliquely across a periodic box and is back where it started after one period. With the
unlimited reconstruction the error of $B_y$ falls by four for every halving of the cells:
second order.

In [ ]:
errors = {}
for cells in (8, 16, 32):
    wave, exact = MHD2D.alfven_wave(cells, limiter="none")
    wave.run(wave.period)
    by = wave.dataset()["magnetic_field_y"]
    errors[cells] = float(np.abs(by - exact["magnetic_field_y"]).mean())
for coarse, fine in ((8, 16), (16, 32)):
    order = np.log2(errors[coarse] / errors[fine])
    print(
        f"{coarse:3d} -> {fine:3d} cells: error {errors[fine]:.2e}, order {order:.2f}"
    )

## 3. A loop across a refined patch

A weak magnetic loop is carried by a uniform flow $(2, 1)$ across a periodic box with a
patch refined by two in its middle. The field of the fine patch comes from the same $A_z$:
the nodes the patch shares with the coarse level carry the fine values, so the flux
through a coarse face equals the sum of the fine fluxes through it, and the flux
register (`patchwork.FluxRegister`) does the same for the mass, momentum and energy.

In [ ]:
cells = 32
hierarchy = AMRHierarchy(
    (2 * cells, cells), ((-1.0, 1.0), (-0.5, 0.5)), periodic=(True, True)
)
hierarchy.add_patch(1, ((cells - 16, cells + 16), (cells - 16, cells + 16)))
loop = MHD2D(hierarchy)
loop.set_state(
    1.0,
    (2.0, 1.0, 0.0),
    1.0,
    vector_potential=lambda x, y: 1e-3 * np.maximum(0.3 - np.hypot(x, y), 0.0),
)
start = loop.totals()


def show(model):
    """The magnetic pressure of the fine patch, over that of the base level."""
    with plasma_plots.figure(1, 1, figsize=(8, 4)) as fig:
        model.dataset(0).magnetic_pressure.plasma.plot.slice(x="x", y="y", ax=fig[0])
        fig[0].plot([-0.5, 0.5, 0.5, -0.5, -0.5], [-0.25, -0.25, 0.25, 0.25, -0.25])


show(loop)

At  = 0.25$ the centre of the loop sits on the upper right corner of the patch:
three quarters of the loop are in coarse cells and the rest on the fine patch, and its
field passes the coarse–fine edges without a kink:

In [ ]:
loop.run(0.25)
show(loop)

In [ ]:
change = {name: loop.totals()[name] - start[name] for name in start}
print(f"max |div B| dx = {loop.divergence():.1e}")
print(f"coarse-fine flux mismatch = {loop.composite_divergence():.1e}")
print({name: f"{value:.1e}" for name, value in change.items()})

## What is next

Resistivity, anisotropic heat conduction, gravity and walls (for the Rayleigh–Taylor,
tearing and magnetothermal instabilities), patches that follow the solution, and levels
that take their own time steps.